**3. Automatic differentiation(autodiff)**

JAX has a good general autodiff system. Computing gradients is a critical part of
modern machine learning methods.

**Taking gradients with** `jax.grad`

In JAX, you can differentiate a scalar-valued function with the `jax.drad()` transformation:

In [ ]:
import jax
import jax . numpy as jnp
from jax import grad
grad_tanh = grad ( jnp . tanh )
print ( grad_tanh (2.0) )

0.070650816


`jax.grad()` takes a function and returns a function. If you have a Python function
f that evaluates the mathematical function , then `jax.grad(f)` is a Python function
that evaluates the mathematical function. That means `grad(f)(x)` represents the
value. Since `jax.grad()` operates on functions, you can apply it to its own output
to differentiate as many times as you like:

In [ ]:
print(grad ( grad ( jnp . tanh ) ) (2.0) )
print(grad ( grad ( grad ( jnp . tanh ) ) ) (2.0) )

-0.13621868
0.25265405


JAX’s autodiff makes it easy to compute higher-order derivatives, because the
functions that compute derivatives are themselves differentiable. Thus, higher-order
derivatives are as easy as stacking transformations. See the example below;
The derivative of $f (x) = x^3 + 2x^2 − 3X + 1$ can be computed as;

In [ ]:
f = lambda x : x **3 + 2* x **2 - 3* x + 1
dfdx = jax . grad ( f )

The higher order derivatives of f are;

$$
\begin{align*}
f'(x) &= 3x^2 + 4x - 3 \\
f''(x) &= 6x + 4 \\
f'''(x) &= 6 \\
f^{iv}(x) &= 0
\end{align*}
$$
In JAX, computing any of these is as easy as chaining the `jax.grad()` function:

In [ ]:
d2fdx = jax.grad(dfdx)
d3fdx = jax.grad(d2fdx)
d4fdx = jax.grad(d3fdx)

# Evaluating for x =1
print ( dfdx (1.) )
print ( d2fdx (1.) )
print ( d3fdx (1.) )
print ( d4fdx (1.) )

4.0
10.0
6.0
0.0


**Computing gradients in a linear logistic regression**

Consider the logistic regression model below;

In [ ]:
key = jax.random.key(0)

def sigmoid(x):
  return 0.5 * (jnp.tanh(x / 2) + 1)

# Outputs probability of a label being true.
def predict(W, b, inputs):
  return sigmoid(jnp.dot(inputs, W) + b)

# Build a toy dataset.
inputs = jnp.array([[0.52, 1.12,  0.77],
                    [0.88, -1.08, 0.15],
                    [0.52, 0.06, -1.30],
                    [0.74, -2.49, 1.39]])
targets = jnp.array([True, True, False, True])

# Training loss is the negative log-likelihood of the training examples.
def loss(W, b):
  preds = predict(W, b, inputs)
  label_probs = preds * targets + (1 - preds) * (1 - targets)
  return -jnp.sum(jnp.log(label_probs))

# Initialize random model coefficients
key, W_key, b_key = jax.random.split(key, 3)
W = jax.random.normal(W_key, (3,))
b = jax.random.normal(b_key, ())

Use the `jax.grad()` function with its argnums argument to differentiate a func-
tion with respect to positional arguments.

In [ ]:
# Differentiate `loss` with respect to the first positional argument:
W_grad = grad(loss, argnums=0)(W, b)
print(f'{W_grad=}')

# Since argnums=0 is the default, this does the same thing:
W_grad = grad(loss)(W, b)
print(f'{W_grad=}')

# But you can choose different values too, and drop the keyword:
b_grad = grad(loss, 1)(W, b)
print(f'{b_grad=}')

# Including tuple values
W_grad, b_grad = grad(loss, (0, 1))(W, b)
print(f'{W_grad=}')
print(f'{b_grad=}')

W_grad=Array([-0.433146 , -0.7354605, -1.2598922], dtype=float32)
W_grad=Array([-0.433146 , -0.7354605, -1.2598922], dtype=float32)
b_grad=Array(-0.6900178, dtype=float32)
W_grad=Array([-0.433146 , -0.7354605, -1.2598922], dtype=float32)
b_grad=Array(-0.6900178, dtype=float32)


**Differentiating with respect to nested lists, tuples, and dicts**

`jax.grad()` can also work with dictionaries for parameters. It returns the output
with respect to each parameter (key) as a separate output dictionary item.

In [ ]:
def loss2(params_dict):
    preds = predict(params_dict['W'], params_dict['b'], inputs)
    label_probs = preds * targets + (1 - preds) * (1 - targets)
    return -jnp.sum(jnp.log(label_probs))

print(grad(loss2)({'W': W, 'b': b}))

{'W': Array([-0.433146 , -0.7354605, -1.2598922], dtype=float32), 'b': Array(-0.6900178, dtype=float32)}


**Evaluating a function and its gradient**

`jax.value_and_grad()` will compute a function’s value as well as its gradient’s value in
one pass.

In [ ]:
loss_value, Wb_grad = jax.value_and_grad(loss, (0, 1))(W, b)
print('loss value', loss_value)
print('loss value', loss(W, b))

loss value 2.9729187
loss value 2.9729187
